# 03 Detector Calibration

Flag rates, score distributions, and false-positive analysis per detector.

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import plotly.express as px
from config import settings
from config.logging_config import configure_logging
from src.data.fetcher import fetch_fx_data
from src.features.builder import build_features
configure_logging(level="WARNING")


In [ ]:
raw = fetch_fx_data(start="2018-01-01")
features = build_features(raw, persist=False)

In [ ]:
from scripts.calibrate import calibrate_thresholds
summary = calibrate_thresholds(features, include_autoencoder=False)
summary

## Score distribution by detector

In [ ]:
from src.detectors import ensemble
outs = ensemble.run_all_detectors(features, include_autoencoder=False)
for name, df in outs.items():
    print(name, "flag_rate=%.4f" % df["anomaly_flag"].mean())

## Threshold sensitivity (z-score)
Sweep the z-score threshold and observe the flag rate.

In [ ]:
from src.detectors import statistical
from config import settings as s
import copy
rates = {}
for thr in [2.5, 3.0, 3.5, 4.0]:
    object.__setattr__(s.DETECTOR, "zscore_threshold", thr)
    out = statistical.zscore_detector(features)
    rates[thr] = float(out["anomaly_flag"].mean())
object.__setattr__(s.DETECTOR, "zscore_threshold", 3.0)
rates